# DATA SPLITTING

In [ ]:
"""
Splits the preprocessed data into four sets:

  - train        : LLM-annotated data for model training
  - val          : LLM-annotated data for hyperparameter tuning
  - test_llm     : LLM-annotated held-out test set
  - test_manual  : Manually coded held-out test set

All splits are stratified to preserve label distribution.
The manual test set is kept entirely separate from the LLM splits, as it serves
as an independent evaluation against human-coded labels
"""

In [1]:
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [2]:
# define paths

IN_LLM      = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/llm_data_clean.parquet"
IN_MANUAL   = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/manual_data_clean.parquet"
OUT_DIR     = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/"

LABEL_COLS  = [f"label_{i}" for i in range(8)]

In [3]:
# load data
df = pd.read_parquet(IN_LLM)
manual_df = pd.read_parquet(IN_MANUAL)

In [4]:
# exclude manually coded articles from the LLM pool to prevent any overlap
# between the manual test set and the LLM train/val/test splits.
manual_keys = set(manual_df["ArticleKey"])
llm_df = df[~df["ArticleKey"].isin(manual_keys)].reset_index(drop=True)

print(f"LLM pool (after removing manual articles): {len(llm_df)}")
print(f"Manual test set:                           {len(manual_df)}")

LLM pool (after removing manual articles): 17477
Manual test set:                           500


In [5]:
# split llm data into train, val and test
# first split: carve out 20% as test_llm
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in msss.split(llm_df, llm_df[LABEL_COLS]):
    train_df = llm_df.iloc[train_idx]
    test_df  = llm_df.iloc[test_idx]

# second split: carve out 20% of the remaining 80% as val (16% of total)
msss_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, val_idx in msss_val.split(train_df, train_df[LABEL_COLS]):
    train_final = train_df.iloc[train_idx]
    val_df      = train_df.iloc[val_idx]


In [6]:
# check label distribution
print("\nLabel distributions (mean per label):")
for name, split in [("Train",       train_final),
                    ("Val",         val_df),
                    ("Test LLM",    test_df),
                    ("Test Manual", manual_df)]:
    print(f"\n  {name} (n={len(split)}):")
    print(split[LABEL_COLS].mean().round(3))


Label distributions (mean per label):

  Train (n=11218):
label_0    0.050
label_1    0.156
label_2    0.026
label_3    0.159
label_4    0.818
label_5    0.019
label_6    0.030
label_7    0.044
dtype: float64

  Val (n=2782):
label_0    0.051
label_1    0.157
label_2    0.026
label_3    0.161
label_4    0.825
label_5    0.019
label_6    0.030
label_7    0.044
dtype: float64

  Test LLM (n=3477):
label_0    0.051
label_1    0.158
label_2    0.026
label_3    0.161
label_4    0.825
label_5    0.019
label_6    0.030
label_7    0.044
dtype: float64

  Test Manual (n=500):
label_0    0.902
label_1    0.002
label_2    0.002
label_3    0.002
label_4    0.082
label_5    0.008
label_6    0.012
label_7    0.010
dtype: float64


In [7]:
# check overlap
splits = {
    "train":       set(train_final["ArticleKey"]),
    "val":         set(val_df["ArticleKey"]),
    "test_llm":    set(test_df["ArticleKey"]),
    "test_manual": set(manual_df["ArticleKey"]),
}
print("\nArticleKey overlap between splits:")
pairs = [
    ("train",    "val"),
    ("train",    "test_llm"),
    ("train",    "test_manual"),
    ("val",      "test_llm"),
    ("val",      "test_manual"),
    ("test_llm", "test_manual"),
]
for a, b in pairs:
    print(f"  {a} ∩ {b}: {len(splits[a] & splits[b])}")


ArticleKey overlap between splits:
  train ∩ val: 0
  train ∩ test_llm: 0
  train ∩ test_manual: 0
  val ∩ test_llm: 0
  val ∩ test_manual: 0
  test_llm ∩ test_manual: 0


In [8]:
# save splits
train_final.to_parquet(OUT_DIR + "train.parquet")
val_df.to_parquet(     OUT_DIR + "val.parquet")
test_df.to_parquet(    OUT_DIR + "test_llm.parquet")
manual_df.to_parquet(  OUT_DIR + "test_manual.parquet")

print("\nAll four splits saved:")
print(f"  train:       {len(train_final)}")
print(f"  val:         {len(val_df)}")
print(f"  test_llm:    {len(test_df)}")
print(f"  test_manual: {len(manual_df)}")


All four splits saved:
  train:       11218
  val:         2782
  test_llm:    3477
  test_manual: 500
